# Node Diffusion Cross-Attention Training (Kaggle)
BERT 文本编码器 + Cross-Attention 的节点坐标扩散生成

In [1]:
import os, shutil
REPO_DIR = '/kaggle/working/PlanDiffusion_wzm'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
os.system(f'git clone -b wzm-shenzhou https://github.com/WeeZHnMin/PlanDiffusion_wzm.git {REPO_DIR}')
print('repo ready:', REPO_DIR)

Cloning into '/kaggle/working/PlanDiffusion_wzm'...


repo ready: /kaggle/working/PlanDiffusion_wzm


In [ ]:
import os, json as _json

# ── 路径配置 ──────────────────────────────────────────────────────────────
BERT_PATH = 'bert-base-uncased'
DATA_PATH = '/kaggle/input/datasets/yahiie/node-diffusion-6k/graph_dataset_6k.npz'
SAVE_DIR  = '/kaggle/working/checkpoints/node_diffusion_cross_att'
os.makedirs(SAVE_DIR, exist_ok=True)

# ── HuggingFace 配置 ───────────────────────────────────────────────────────
# Kaggle Notebook → Add-ons → Secrets → key=HF_TOKEN
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

HF_REPO_ID = 'wzmmmm/plandiff-adj-cross-6k'
print(f'HF_REPO_ID : {HF_REPO_ID}')
print(f'HF_TOKEN   : {"OK (" + HF_TOKEN[:8] + "...)" if HF_TOKEN else "未设置，不推送/不拉取"}')

# ── 自动从 HF 拉取最新权重作为断点续练 ────────────────────────────────────
RESUME = ''
if HF_TOKEN:
    try:
        from huggingface_hub import hf_hub_download
        _local = hf_hub_download(
            repo_id   = HF_REPO_ID,
            filename  = 'latest.pt',
            token     = HF_TOKEN,
            local_dir = SAVE_DIR,
        )
        RESUME = _local
        print(f'自动恢复: 已拉取 {HF_REPO_ID}/latest.pt → {RESUME}')
    except Exception as _e:
        print(f'HF 上暂无权重，从头训练 ({_e})')

# ── 训练超参数 ────────────────────────────────────────────────────────────
BATCH_SIZE      = 384
TOTAL_STEPS     = 250000
LR              = 1e-4
WEIGHT_DECAY    = 1e-4
LOG_INTERVAL    = 100
SAVE_INTERVAL   = 1000
TIMESTEPS       = 1000

# ── 模型参数 ──────────────────────────────────────────────────────────────
MODEL_CHANNELS  = 384
NUM_LAYERS      = 6
NUM_HEADS       = 6
UNFREEZE_LAYERS = 0

In [3]:
import os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
os.makedirs(SAVE_DIR, exist_ok=True)

device: cuda


In [4]:
if not os.path.exists(DATA_PATH):
    print('构建 BERT NPZ ...')
    from transformers import BertTokenizer
    import json, random, time
    tokenizer = BertTokenizer.from_pretrained(BERT_PATH)

    MAX_NODES    = 40
    MAX_TEXT_LEN = 192

    adj_list = []; mask_list = []; ids_list = []; coords_list = []
    ptok_list = []; pmask_list = []; plen_list = []; nnodes_list = []
    rng = random.Random(42)
    AUGMENT = 8
    n_skipped = 0

    def permute_graph(adj, combo_ids, coords, n, perm):
        full_perm = perm + list(range(n, MAX_NODES))
        return adj[np.ix_(full_perm, full_perm)], combo_ids[full_perm], coords[full_perm]

    t0 = time.perf_counter()
    n_graphs = 0
    with open(JSONL_PATH, encoding='utf-8') as f:
        for line_no, line in enumerate(f):
            line = line.strip()
            if not line: continue
            rec    = json.loads(line)
            prompt = rec.get('prompt', '').replace('\n', ' ').strip()

            enc = tokenizer(prompt, add_special_tokens=True)
            if len(enc['input_ids']) > MAX_TEXT_LEN:
                n_skipped += 1
                continue

            n = int(rec['n_nodes'])
            n_graphs += 1

            adj_full = np.array(rec['adj_matrix'], dtype=np.int32)
            np.fill_diagonal(adj_full, 0)
            combo_ids = np.array(rec['node_combo_ids'][:MAX_NODES], dtype=np.int32)
            raw_coords = rec['node_coords'][:MAX_NODES]
            coords = np.zeros((MAX_NODES, 2), dtype=np.int32)
            coords[:len(raw_coords)] = raw_coords
            mask = np.zeros(MAX_NODES, dtype=np.int32)
            mask[:n] = 1

            padded   = np.zeros(MAX_TEXT_LEN, dtype=np.int32)
            attn_msk = np.zeros(MAX_TEXT_LEN, dtype=np.int32)
            tlen = len(enc['input_ids'])
            padded[:tlen]   = enc['input_ids']
            attn_msk[:tlen] = enc['attention_mask']

            base_perm = list(range(n))
            perms = [base_perm]
            for _ in range(AUGMENT - 1):
                p = base_perm[:]; rng.shuffle(p); perms.append(p)

            for perm in perms:
                new_adj, new_ids, new_coords = permute_graph(adj_full, combo_ids, coords, n, perm)
                adj_list.append(new_adj); mask_list.append(mask)
                ids_list.append(new_ids); coords_list.append(new_coords)
                ptok_list.append(padded); pmask_list.append(attn_msk)
                plen_list.append(tlen); nnodes_list.append(n)

            if (line_no + 1) % 10000 == 0:
                print(f'  {line_no+1} 张图 → {len(adj_list)} 条记录  ({time.perf_counter()-t0:.1f}s)')

    print(f'共 {n_graphs} 张图（跳过 {n_skipped} 条 >{MAX_TEXT_LEN} tokens），增强后 {len(adj_list)} 条记录，保存中 ...')
    np.savez_compressed(
        DATA_PATH,
        adj_matrix     = np.stack(adj_list),
        node_mask      = np.stack(mask_list),
        node_combo_ids = np.stack(ids_list),
        node_coords    = np.stack(coords_list),
        prompt_tokens  = np.stack(ptok_list),
        prompt_mask    = np.stack(pmask_list),
        prompt_lens    = np.array(plen_list, dtype=np.int32),
        n_nodes        = np.array(nnodes_list, dtype=np.int32),
    )
    print(f'saved → {DATA_PATH}  ({time.perf_counter()-t0:.1f}s)')
else:
    print(f'NPZ 已存在，跳过构建: {DATA_PATH}')

NPZ 已存在，跳过构建: /kaggle/input/datasets/wzmmmm/node-diffusion-6k/graph_dataset_6k.npz


In [5]:
def timestep_embedding(timesteps, dim):
    half = dim // 2
    freqs = torch.exp(
        -math.log(10000) * torch.arange(half, dtype=torch.float32, device=timesteps.device) / half
    )
    args = timesteps[:, None].float() * freqs[None]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)


def attention(q, k, v, d_k, mask=None, dropout=None):
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask.unsqueeze(1) == 1, -1e4)
    scores = F.softmax(scores.float(), dim=-1).to(q.dtype)
    if dropout is not None:
        scores = dropout(scores)
    return torch.matmul(scores, v)


class MultiHeadAttention(nn.Module):
    def __init__(self, heads, d_model, dropout=0.1):
        super().__init__()
        self.d_k = d_model // heads
        self.h = heads
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        bs = q.size(0)
        q = self.q_linear(q).view(bs, -1, self.h, self.d_k).transpose(1, 2)
        k = self.k_linear(k).view(bs, -1, self.h, self.d_k).transpose(1, 2)
        v = self.v_linear(v).view(bs, -1, self.h, self.d_k).transpose(1, 2)
        out = attention(q, k, v, self.d_k, mask, self.dropout)
        out = out.transpose(1, 2).contiguous().view(bs, -1, self.h * self.d_k)
        return self.out(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_model * 2)
        self.linear2 = nn.Linear(d_model * 2, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))


class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        self.norm1      = nn.LayerNorm(d_model)
        self.norm_cross = nn.LayerNorm(d_model)
        self.norm2      = nn.LayerNorm(d_model)
        self.adj_attn   = MultiHeadAttention(heads, d_model, dropout)
        self.cross_attn = MultiHeadAttention(heads, d_model, dropout)
        self.ff         = FeedForward(d_model, dropout)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x, adj_mask, text_feat, text_mask):
        x2 = self.norm1(x)
        x  = x + self.dropout(self.adj_attn(x2, x2, x2, adj_mask))
        x2 = self.norm_cross(x)
        x  = x + self.dropout(self.cross_attn(x2, text_feat, text_feat, text_mask))
        x2 = self.norm2(x)
        x  = x + self.dropout(self.ff(x2))
        return x


class NodeDiffusionTransformer(nn.Module):
    def __init__(self, model_channels=384, num_layers=6, num_heads=6,
                 dropout=0.1, bert_name='bert-base-uncased', unfreeze_layers=0):
        super().__init__()
        self.model_channels = model_channels
        self.time_embed = nn.Sequential(
            nn.Linear(model_channels, model_channels),
            nn.SiLU(),
            nn.Linear(model_channels, model_channels),
        )
        self.input_emb = nn.Linear(2, model_channels)

        self.bert = BertModel.from_pretrained(bert_name)
        for p in self.bert.parameters():
            p.requires_grad = False
        if unfreeze_layers > 0:
            n_layers = len(self.bert.encoder.layer)
            for layer in self.bert.encoder.layer[n_layers - unfreeze_layers:]:
                for p in layer.parameters():
                    p.requires_grad = True
        self.text_proj = nn.Linear(self.bert.config.hidden_size, model_channels)

        self.layers = nn.ModuleList(
            [EncoderLayer(model_channels, num_heads, dropout) for _ in range(num_layers)]
        )
        self.coord_head = nn.Sequential(
            nn.Linear(model_channels, model_channels),
            nn.ReLU(),
            nn.Linear(model_channels, model_channels // 2),
            nn.Linear(model_channels // 2, 2),
        )
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.parameters())
        print(f'NodeDiffusionTransformer: {trainable:,} trainable / {total:,} total')

    def _build_adj_mask(self, adj_matrix, node_mask):
        adj_mask = 1 - adj_matrix
        pad_keys = (1 - node_mask).unsqueeze(1)
        return torch.clamp(adj_mask + pad_keys, 0, 1)

    def forward(self, x, timesteps, adj_matrix, node_mask,
                prompt_tokens=None, prompt_mask=None, **kwargs):
        del kwargs
        B, _, N = x.shape
        x = x.permute(0, 2, 1).float()
        t_emb    = self.time_embed(timestep_embedding(timesteps, self.model_channels)).unsqueeze(1)
        node_emb = self.input_emb(x) + t_emb
        adj_mask = self._build_adj_mask(adj_matrix.float(), node_mask.float())

        if prompt_tokens is not None:
            bert_attn = prompt_mask if prompt_mask is not None else (prompt_tokens != 0).long()
            with torch.no_grad():
                text_hidden = self.bert(input_ids=prompt_tokens,
                                        attention_mask=bert_attn).last_hidden_state
            text_feat = self.text_proj(text_hidden)
            text_mask = (1 - bert_attn.float()).unsqueeze(1)
        else:
            text_feat = torch.zeros(B, 1, self.model_channels, device=node_emb.device, dtype=node_emb.dtype)
            text_mask = None

        seq = node_emb
        for layer in self.layers:
            seq = layer(seq, adj_mask, text_feat, text_mask)
        return self.coord_head(seq).permute(0, 2, 1)

## 扩散过程

In [6]:
class GaussianDiffusion:
    """Cosine noise schedule (Nichol & Dhariwal 2021)."""
    def __init__(self, timesteps=1000):
        self.T = timesteps
        t          = torch.arange(timesteps + 1) / timesteps
        f          = torch.cos((t + 0.008) / 1.008 * math.pi / 2) ** 2
        alphas_bar = f / f[0]
        betas      = (1 - alphas_bar[1:] / alphas_bar[:-1]).clamp(max=0.999)
        alphas_bar = alphas_bar[1:]
        alphas          = 1.0 - betas
        alphas_bar_prev = torch.cat([torch.tensor([1.0]), alphas_bar[:-1]])
        self.betas                     = betas
        self.alphas                    = alphas
        self.alphas_bar                = alphas_bar
        self.alphas_bar_prev           = alphas_bar_prev
        self.sqrt_alphas_bar           = alphas_bar.sqrt()
        self.sqrt_one_minus_alphas_bar = (1 - alphas_bar).sqrt()
        self.posterior_variance = (betas * (1 - alphas_bar_prev) / (1 - alphas_bar)).clamp(min=1e-20)

    def _to(self, device):
        for attr in ['betas','alphas','alphas_bar','alphas_bar_prev',
                     'sqrt_alphas_bar','sqrt_one_minus_alphas_bar','posterior_variance']:
            setattr(self, attr, getattr(self, attr).to(device))
        return self

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        s1 = self.sqrt_alphas_bar[t].view(-1, 1, 1)
        s2 = self.sqrt_one_minus_alphas_bar[t].view(-1, 1, 1)
        return s1 * x0 + s2 * noise, noise

    def training_losses(self, model, x0, t, model_kwargs):
        self._to(x0.device)
        x0 = x0.float()
        coord_noise = torch.randn_like(x0)
        xt, _       = self.q_sample(x0, t, coord_noise)
        pred_coord_noise = model(xt, t, **model_kwargs)
        node_mask  = model_kwargs['node_mask'].float()
        coord_mask = node_mask.unsqueeze(1)
        coord_loss = ((pred_coord_noise - coord_noise) ** 2 * coord_mask).sum() / (coord_mask.sum() * 2 + 1e-8)
        with torch.no_grad():
            s1 = self.sqrt_alphas_bar[t].view(-1, 1, 1)
            s2 = self.sqrt_one_minus_alphas_bar[t].view(-1, 1, 1)
            pred_x0    = (xt - s2 * pred_coord_noise.float()) / s1.clamp(min=1e-3)
            raw_mse    = ((pred_x0 - x0) ** 2 * coord_mask).sum() / (coord_mask.sum() * 2 + 1e-8)
            coord_rmse = raw_mse.sqrt().item()
        return coord_loss, coord_rmse

## 数据集

In [7]:
class NodeDataset(Dataset):
    def __init__(self, npz_path):
        d = np.load(npz_path, allow_pickle=True)
        self.coords        = d['node_coords'].astype(np.float32)
        self.adj_matrix    = d['adj_matrix'].astype(np.float32)
        self.node_mask     = d['node_mask'].astype(np.float32)
        self.node_types    = d['node_combo_ids'].astype(np.int64)
        self.prompt_tokens = d['prompt_tokens'].astype(np.int64)
        self.prompt_mask   = d['prompt_mask'].astype(np.float32)  # BERT attention_mask
        print(f'NodeDataset: {len(self.coords)} samples')

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        x = self.coords[idx].T.copy()
        cond = {
            'adj_matrix':    self.adj_matrix[idx],
            'node_mask':     self.node_mask[idx],
            'node_types':    self.node_types[idx],
            'prompt_tokens': self.prompt_tokens[idx],
            'prompt_mask':   self.prompt_mask[idx],
        }
        return torch.from_numpy(x), {k: torch.from_numpy(v) for k, v in cond.items()}


def make_loader(npz_path, batch_size):
    ds = NodeDataset(npz_path)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True,
                        num_workers=4, drop_last=True, pin_memory=True)
    def infinite():
        while True:
            yield from loader
    return infinite()

## 初始化模型

In [8]:
model = NodeDiffusionTransformer(
    model_channels  = MODEL_CHANNELS,
    num_layers      = NUM_LAYERS,
    num_heads       = NUM_HEADS,
    bert_name       = BERT_PATH,
    unfreeze_layers = UNFREEZE_LAYERS,
).to(device)

if torch.cuda.device_count() > 1:
    print(f'使用 {torch.cuda.device_count()} 个 GPU')
    model = torch.nn.DataParallel(model)

diffusion = GaussianDiffusion(timesteps=TIMESTEPS)
opt       = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=WEIGHT_DECAY,
)
scaler = torch.amp.GradScaler('cuda')

start_step = 0
if RESUME:
    ckpt = torch.load(RESUME, map_location=device)
    raw_state   = ckpt['model']
    is_dp_model = isinstance(model, torch.nn.DataParallel)
    has_dp_keys = any(k.startswith('module.') for k in raw_state)
    if is_dp_model and not has_dp_keys:
        raw_state = {'module.' + k: v for k, v in raw_state.items()}
    elif not is_dp_model and has_dp_keys:
        raw_state = {k[7:]: v for k, v in raw_state.items()}
    missing, unexpected = model.load_state_dict(raw_state, strict=False)
    if missing:    print(f'  missing keys: {missing}')
    if unexpected: print(f'  unexpected keys: {unexpected}')
    if not unexpected:
        opt.load_state_dict(ckpt['opt'])
        if 'scaler' in ckpt: scaler.load_state_dict(ckpt['scaler'])
    else:
        print('  架构已变更，优化器从头初始化')
    start_step = ckpt['step'] + 1
    print(f'resumed from step {start_step}')
    if start_step >= TOTAL_STEPS:
        TOTAL_STEPS = start_step + 200000

data = make_loader(DATA_PATH, BATCH_SIZE)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NodeDiffusionTransformer: 11,470,274 trainable / 120,952,514 total
使用 2 个 GPU
resumed from step 20001
NodeDataset: 47424 samples


## 训练循环

In [ ]:
import threading
from huggingface_hub import HfApi

def _do_push_hf(ckpt_path, step, repo_id, token):
    try:
        api = HfApi(token=token)
        try:
            api.delete_repo(repo_id=repo_id, repo_type="model")
        except Exception:
            pass
        api.create_repo(repo_id, private=True, repo_type="model")
        api.upload_file(
            path_or_fileobj=ckpt_path,
            path_in_repo="latest.pt",
            repo_id=repo_id,
            commit_message=f"step {step}",
        )
        print(f"  [HF] step={step} -> {repo_id}/latest.pt uploaded")
    except Exception as e:
        print(f"  [HF] push failed: {e}")

def push_to_hf(save_dir, step, repo_id, token):
    if not repo_id or not token:
        return
    ckpt_path = os.path.join(save_dir, "latest.pt")
    if not os.path.exists(ckpt_path):
        return
    t = threading.Thread(target=_do_push_hf,
                         args=(ckpt_path, step, repo_id, token), daemon=True)
    t.start()


model.train()
running_loss = running_rmse = 0.0

for step in range(start_step, TOTAL_STEPS):
    x, cond = next(data)
    x    = x.to(device)
    cond = {k: v.to(device) for k, v in cond.items()}
    t    = torch.randint(0, TIMESTEPS, (x.shape[0],), device=device)

    opt.zero_grad()
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        loss, coord_rmse = diffusion.training_losses(model, x, t, cond)

    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], 1.0
    )
    scaler.step(opt)
    scaler.update()

    running_loss += loss.item()
    running_rmse += coord_rmse

    if step % LOG_INTERVAL == 0:
        n = LOG_INTERVAL if step > 0 else 1
        print(f'step {step:6d} | loss {running_loss/n:.4f} | coord_rmse {running_rmse/n:.2f} px')
        running_loss = running_rmse = 0.0

    if step > 0 and step % SAVE_INTERVAL == 0:
        path = os.path.join(SAVE_DIR, 'latest.pt')
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                    'scaler': scaler.state_dict(), 'step': step}, path)
        print(f'  saved → {path}')
        push_to_hf(SAVE_DIR, step, HF_REPO_ID, HF_TOKEN)

torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
            'scaler': scaler.state_dict(), 'step': TOTAL_STEPS},
           os.path.join(SAVE_DIR, 'latest.pt'))
push_to_hf(SAVE_DIR, TOTAL_STEPS, HF_REPO_ID, HF_TOKEN)
print('训练完成')

step  20100 | loss 0.5134 | coord_rmse 7.30 px
